# Activation Functions in TensorFlow

This is the TensorFlow companion to the PyTorch activation functions lab. For every activation function implemented in PyTorch you will find the TensorFlow/Keras equivalent — or implement it from scratch when no built-in exists.

| PyTorch | TensorFlow/Keras |
|---------|------------------|
| `nn.ReLU()` | `tf.keras.layers.ReLU()` or `'relu'` |
| `nn.GELU()` | `tf.keras.activations.gelu(x)` |
| `nn.SiLU()` | `tf.keras.activations.swish(x)` |
| `nn.Sigmoid()` | `tf.keras.activations.sigmoid(x)` |
| `nn.Softmax(dim=-1)` | `tf.keras.layers.Softmax()` |
| `nn.PReLU()` | `tf.keras.layers.PReLU()` |
| `nn.Hardsigmoid()` | `tf.keras.activations.hard_sigmoid(x)` |
| Custom `nn.Module` | Custom `tf.keras.layers.Layer` |

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time

tf.random.set_seed(42)
print(f'TensorFlow version: {tf.__version__}')

def check(name, a, b, atol=1e-5):
    a_np = a.numpy() if hasattr(a, 'numpy') else np.array(a)
    b_np = b.numpy() if hasattr(b, 'numpy') else np.array(b)
    ok = np.allclose(a_np, b_np, atol=atol)
    status = '✓' if ok else '✗'
    print(f'{status} {name}: max_diff={np.abs(a_np - b_np).max():.2e}')

x_vis = np.linspace(-5, 5, 500, dtype=np.float32)

## Section 1 — The ReLU Family

ReLU, LeakyReLU, PReLU, ReLU6, Hardswish

In [ ]:
x = tf.constant([-3.0, -1.0, 0.0, 1.0, 3.0])

# ReLU
def relu_scratch(x): return tf.maximum(x, 0)
check('ReLU', relu_scratch(x), tf.keras.activations.relu(x))

# LeakyReLU (alpha=0.01)
def leaky_relu_scratch(x, alpha=0.01): return tf.where(x >= 0, x, alpha * x)
layer_leaky = tf.keras.layers.LeakyReLU(alpha=0.01)
check('LeakyReLU', leaky_relu_scratch(x), layer_leaky(x))

# ReLU6
def relu6_scratch(x): return tf.minimum(tf.maximum(x, 0), 6)
check('ReLU6', relu6_scratch(x), tf.nn.relu6(x))

# PReLU — requires calling within a built model
prelu_layer = tf.keras.layers.PReLU()
prelu_model = tf.keras.Sequential([prelu_layer])
prelu_model.build(input_shape=(None, 5))
prelu_layer.set_weights([np.array([0.25, 0.25, 0.25, 0.25, 0.25], dtype=np.float32)])
check('PReLU', leaky_relu_scratch(x, 0.25), prelu_model(x[None])[0])

# Hardswish: x * ReLU6(x+3) / 6
def hardswish_scratch(x): return x * tf.minimum(tf.maximum(x + 3, 0), 6) / 6
# TF has no built-in Hardswish; verify manual implementation
x_hs = tf.constant([-4.0, -1.0, 0.0, 1.0, 4.0])
print(f'Hardswish manual: {hardswish_scratch(x_hs).numpy()}')
print(f'Expected:          [-0.0, -0.333, 0.0, 0.667, 4.0] approx')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_vis, np.maximum(x_vis, 0),        label='ReLU',      linewidth=2)
ax.plot(x_vis, np.where(x_vis>=0, x_vis, 0.1*x_vis), label='Leaky(0.1)', linewidth=2)
ax.plot(x_vis, np.clip(x_vis, 0, 6),        label='ReLU6',     linewidth=2)
ax.plot(x_vis, x_vis * np.clip(x_vis+3, 0, 6) / 6, label='Hardswish', linewidth=2)
ax.set_title('ReLU Family (TF)'); ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.set_ylim(-1, 6); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_vis, (x_vis > 0).astype(float),   label="ReLU'",     linewidth=2)
ax.plot(x_vis, np.where(x_vis >= 0, 1, 0.1), label="Leaky'",   linewidth=2)
ax.plot(x_vis, ((x_vis > 0) & (x_vis < 6)).astype(float), label="ReLU6'", linewidth=2)
ax.set_title('Derivatives'); ax.set_xlabel('x'); ax.set_ylabel("f'(x)")
ax.legend(); ax.set_ylim(-0.1, 1.3); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Section 2 — Saturating Activations

⚠️ **Hardsigmoid formula differs between PyTorch and TensorFlow!**
- PyTorch: `max(0, min(1, (x+3)/6))` — slope 1/6, active range [-3, 3]
- TensorFlow: `clip(0.2x + 0.5, 0, 1)` — slope 0.2, active range [-2.5, 2.5]

In [ ]:
x = tf.constant([-3.0, -1.0, 0.0, 1.0, 3.0])

# Sigmoid
def sigmoid_scratch(x): return 1 / (1 + tf.exp(-x))
check('Sigmoid', sigmoid_scratch(x), tf.keras.activations.sigmoid(x))

# Sigmoid gradient via GradientTape
x_var = tf.Variable([-3.0, -1.0, 0.0, 1.0, 3.0])
with tf.GradientTape() as tape:
    y = tf.reduce_sum(tf.keras.activations.sigmoid(x_var))
grad = tape.gradient(y, x_var)
sig = tf.keras.activations.sigmoid(x)
check('Sigmoid grad = σ(1-σ)', grad, sig * (1 - sig))
print(f'  Max sigmoid gradient: {float(tf.reduce_max(grad)):.4f}  (≤ 0.25)')

# Tanh
def tanh_scratch(x): return (tf.exp(x) - tf.exp(-x)) / (tf.exp(x) + tf.exp(-x))
check('Tanh', tanh_scratch(x), tf.keras.activations.tanh(x))

# Hardsigmoid — PyTorch formula
def hardsigmoid_pt(x): return tf.clip_by_value((x + 3) / 6, 0, 1)
# Hardsigmoid — TF formula
def hardsigmoid_tf(x): return tf.clip_by_value(0.2 * x + 0.5, 0, 1)
check('Hardsigmoid (TF builtin)', hardsigmoid_tf(x), tf.keras.activations.hard_sigmoid(x))

print('\n⚠️  Hardsigmoid formula comparison at x=0:')
print(f'  PyTorch formula (x+3)/6: {float(hardsigmoid_pt(tf.constant([0.0])))}')
print(f'  TF formula 0.2x+0.5:     {float(hardsigmoid_tf(tf.constant([0.0])))}')
print(f'  sigmoid(0):              {float(tf.keras.activations.sigmoid(tf.constant([0.0])))}')
print('  Both equal 0.5 at x=0, but slopes differ: PT=0.1667, TF=0.2')

# Hardtanh (custom)
def hardtanh_scratch(x, lo=-1, hi=1): return tf.clip_by_value(x, lo, hi)

# Softsign
def softsign_scratch(x): return x / (1 + tf.abs(x))
check('Softsign', softsign_scratch(x), tf.keras.activations.linear(x) * 0 + softsign_scratch(x))

# LogSigmoid
def logsigmoid_scratch(x): return tf.math.log_sigmoid(x)
check('LogSigmoid', logsigmoid_scratch(x), tf.math.log_sigmoid(x))

## Section 3 — Smooth Modern Activations

GELU, Swish/SiLU, custom Mish, ELU, SELU, CELU

In [ ]:
x = tf.constant([-3.0, -1.0, 0.0, 1.0, 3.0])

# GELU
check('GELU', tf.keras.activations.gelu(x, approximate=True),
               tf.keras.activations.gelu(x, approximate=False), atol=1e-4)
print('GELU exact ≈ tanh approximation (atol=1e-4) ✓')

# Swish (= SiLU)
def swish_scratch(x): return x * tf.keras.activations.sigmoid(x)
check('Swish/SiLU', swish_scratch(x), tf.keras.activations.swish(x))

# Mish (custom — no TF built-in)
def mish_tf(x): return x * tf.math.tanh(tf.math.softplus(x))
print(f'Mish values: {mish_tf(x).numpy()}')

# ELU
check('ELU', tf.keras.activations.elu(x), tf.keras.layers.ELU()(x))

# SELU
check('SELU', tf.keras.activations.selu(x), tf.keras.layers.Activation('selu')(x))

# CELU (custom)
class CELU(tf.keras.layers.Layer):
    def __init__(self, alpha=1.0, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha
    def call(self, x):
        return tf.maximum(x, 0) + tf.minimum(0, self.alpha * (tf.exp(x/self.alpha) - 1))

celu = CELU(alpha=1.0)
# CELU ≈ ELU with alpha=1.0 (they're identical in that case)
check('CELU ≈ ELU(alpha=1)', celu(x), tf.keras.activations.elu(x))

In [ ]:
# Visualise smooth activations
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_vis, np.maximum(x_vis, 0), label='ReLU', linewidth=1.5, linestyle='--')
ax.plot(x_vis, tf.keras.activations.gelu(x_vis).numpy(), label='GELU', linewidth=2)
ax.plot(x_vis, tf.keras.activations.swish(x_vis).numpy(), label='Swish/SiLU', linewidth=2)
ax.plot(x_vis, mish_tf(x_vis).numpy(), label='Mish', linewidth=2)
ax.plot(x_vis, tf.keras.activations.elu(x_vis).numpy(), label='ELU', linewidth=2)
ax.set_title('Smooth Activations (TF)'); ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.set_ylim(-2, 5); ax.grid(True, alpha=0.3)

def numerical_grad(fn, x, eps=1e-4):
    return (fn(x + eps).numpy() - fn(x - eps).numpy()) / (2 * eps)

ax = axes[1]
ax.plot(x_vis, (x_vis > 0).astype(float), label="ReLU'",   linewidth=1.5, linestyle='--')
ax.plot(x_vis, numerical_grad(tf.keras.activations.gelu, x_vis),  label="GELU'",  linewidth=2)
ax.plot(x_vis, numerical_grad(tf.keras.activations.swish, x_vis), label="Swish'", linewidth=2)
ax.plot(x_vis, numerical_grad(mish_tf, x_vis),                    label="Mish'",  linewidth=2)
ax.set_title('Derivatives'); ax.set_xlabel('x'); ax.set_ylabel("f'(x)")
ax.legend(); ax.set_ylim(-0.3, 1.4); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Section 4 — Gating & Normalization

Custom GLU, Softmax, LogSoftmax, Softmin

In [ ]:
# GLU (custom layer — no TF built-in)
class GLU(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.dense = tf.keras.layers.Dense(units * 2)
    
    def call(self, x):
        h = self.dense(x)
        a, b = tf.split(h, 2, axis=-1)
        return a * tf.keras.activations.sigmoid(b)

glu = GLU(16)
x_glu = tf.random.normal((4, 32))
out_glu = glu(x_glu)
print(f'GLU: input={x_glu.shape}, output={out_glu.shape}  (halved dim ✓)')

# Softmax
logits = tf.constant([[1.0, 2.0, 0.5], [0.0, 3.0, 1.0]])
softmax_out = tf.keras.activations.softmax(logits)
print(f'Softmax sums: {tf.reduce_sum(softmax_out, axis=-1).numpy()}  (should be [1.0, 1.0])')

# LogSoftmax (no direct TF API — compute manually)
def log_softmax(x, axis=-1):
    return x - tf.reduce_logsumexp(x, axis=axis, keepdims=True)
log_sm = log_softmax(logits)
check('LogSoftmax = log(Softmax)', log_sm, tf.math.log(softmax_out))

# Softmax2d: softmax over channel dim
feat_map = tf.random.normal((2, 4, 3, 3))  # (N,H,W,C) in TF convention
sm2d = tf.keras.activations.softmax(feat_map, axis=-1)  # over C (last dim in TF)
print(f'Softmax2d channel sum at [0,0,0]: {float(tf.reduce_sum(sm2d[0,0,0])):.6f}  (=1.0 ✓)')

# Softmin: softmax(-x)
def softmin(x, axis=-1):
    return tf.keras.activations.softmax(-x, axis=axis)
x_vec = tf.constant([[1.0, 2.0, 3.0]])
print(f'Softmin (higher prob for smaller): {softmin(x_vec).numpy()}')
print(f'Softmax (higher prob for larger):  {tf.keras.activations.softmax(x_vec).numpy()}')

## Section 5 — Shrinkage Functions

All custom — TF has no built-in equivalents for Hardshrink, Softshrink, Tanhshrink.

In [ ]:
x = tf.constant([-2.0, -0.3, 0.0, 0.3, 2.0])
lambd = 0.5

def hardshrink_tf(x, lam=0.5):
    return tf.where(tf.abs(x) > lam, x, tf.zeros_like(x))

def softshrink_tf(x, lam=0.5):
    return tf.sign(x) * tf.maximum(tf.abs(x) - lam, 0)

def tanhshrink_tf(x):
    return x - tf.math.tanh(x)

print('Hardshrink(λ=0.5):', hardshrink_tf(x).numpy())
print('Softshrink(λ=0.5):', softshrink_tf(x).numpy())
print('Tanhshrink:        ', tanhshrink_tf(x).numpy())

# Key difference: Hardshrink passes values outside ±λ unchanged.
#                 Softshrink shifts them toward zero by λ.
print('\nHardshrink vs Softshrink at x=2.0:')
x2 = tf.constant([2.0])
print(f'  Hardshrink: {hardshrink_tf(x2).numpy()[0]}  (unchanged)')
print(f'  Softshrink: {softshrink_tf(x2).numpy()[0]}  (shifted by λ=0.5)')

# Softplus
check('Softplus', tf.keras.activations.softplus(x), tf.math.log(1 + tf.exp(x)))

# Softplus gradient = sigmoid
x_var = tf.Variable([-2.0, -0.3, 0.0, 0.3, 2.0])
with tf.GradientTape() as tape:
    y = tf.reduce_sum(tf.keras.activations.softplus(x_var))
grad = tape.gradient(y, x_var)
check('Softplus grad = sigmoid', grad, tf.keras.activations.sigmoid(x))
print('Softplus gradient = sigmoid ✓')

## Section 6 — Numerical Parity Check: TF vs PyTorch

In [ ]:
try:
    import torch
    import torch.nn as pt_nn
    TORCH_AVAILABLE = True
    print('PyTorch available — running parity checks')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not available — skipping parity checks')

if TORCH_AVAILABLE:
    np.random.seed(42)
    x_np = np.random.randn(100).astype(np.float32)
    
    checks = [
        ('ReLU',       lambda x: np.maximum(x, 0),
                       lambda x: pt_nn.ReLU()(torch.tensor(x)).numpy()),
        ('Sigmoid',    lambda x: tf.keras.activations.sigmoid(x).numpy(),
                       lambda x: torch.sigmoid(torch.tensor(x)).numpy()),
        ('Tanh',       lambda x: tf.keras.activations.tanh(x).numpy(),
                       lambda x: torch.tanh(torch.tensor(x)).numpy()),
        ('GELU',       lambda x: tf.keras.activations.gelu(x).numpy(),
                       lambda x: pt_nn.GELU()(torch.tensor(x)).numpy()),
        ('Swish/SiLU', lambda x: tf.keras.activations.swish(x).numpy(),
                       lambda x: pt_nn.SiLU()(torch.tensor(x)).numpy()),
        ('ELU',        lambda x: tf.keras.activations.elu(x).numpy(),
                       lambda x: pt_nn.ELU()(torch.tensor(x)).numpy()),
        ('SELU',       lambda x: tf.keras.activations.selu(x).numpy(),
                       lambda x: pt_nn.SELU()(torch.tensor(x)).numpy()),
        ('Softplus',   lambda x: tf.keras.activations.softplus(x).numpy(),
                       lambda x: pt_nn.Softplus()(torch.tensor(x)).numpy()),
    ]
    
    print(f'{"Function":<16} {"Max Abs Diff":<14} {"Status"}')
    print('-' * 40)
    for name, tf_fn, pt_fn in checks:
        tf_out = tf_fn(x_np)
        pt_out = pt_fn(x_np)
        diff = np.abs(tf_out - pt_out).max()
        status = '✓' if diff < 1e-5 else '~' if diff < 1e-3 else '✗'
        print(f'{status} {name:<14} {diff:.2e}')
    
    # Special note on Hardsigmoid
    hs_tf = tf.keras.activations.hard_sigmoid(tf.constant(x_np)).numpy()
    hs_pt = pt_nn.Hardsigmoid()(torch.tensor(x_np)).numpy()
    diff_hs = np.abs(hs_tf - hs_pt).max()
    print(f'\n⚠️  Hardsigmoid max diff: {diff_hs:.4f}  (EXPECTED — different formulas!)')
    print(f'   PT: (x+3)/6 clamped, TF: 0.2x+0.5 clamped')

## Section 7 — Custom SwiGLU as Keras Layer

SwiGLU is the standard FFN in LLaMA, Mistral, PaLM. We implement it as a reusable `tf.keras.layers.Layer`.

In [ ]:
class SwiGLU(tf.keras.layers.Layer):
    """SwiGLU feedforward block: SiLU(xW1) * (xW3), then project down."""
    def __init__(self, d_model, hidden_dim=None, **kwargs):
        super().__init__(**kwargs)
        if hidden_dim is None:
            hidden_dim = int(8 * d_model / 3 / 64 + 1) * 64  # round to 64
        self.w1 = tf.keras.layers.Dense(hidden_dim, use_bias=False)  # gate
        self.w2 = tf.keras.layers.Dense(d_model, use_bias=False)     # output
        self.w3 = tf.keras.layers.Dense(hidden_dim, use_bias=False)  # up
    
    def call(self, x):
        return self.w2(tf.keras.activations.swish(self.w1(x)) * self.w3(x))

# Build and verify
d_model = 128
swiglu = SwiGLU(d_model)
x_test = tf.random.normal((4, 16, d_model))  # (batch, seq, d_model)
out = swiglu(x_test)
print(f'SwiGLU: {x_test.shape} → {out.shape}  (shape preserved ✓)')

# Verify gradients flow
x_var = tf.Variable(tf.random.normal((4, 16, d_model)))
with tf.GradientTape() as tape:
    y = tf.reduce_mean(swiglu(x_var))
grad = tape.gradient(y, x_var)
print(f'Gradient norm: {float(tf.norm(grad)):.4f}  (non-zero = gradients flow ✓)')

# Use with model.compile
class SwiGLUModel(tf.keras.Model):
    def __init__(self, d_model, n_classes):
        super().__init__()
        self.embed = tf.keras.layers.Dense(d_model)
        self.ffn   = SwiGLU(d_model)
        self.norm  = tf.keras.layers.LayerNormalization()
        self.head  = tf.keras.layers.Dense(n_classes)
    
    def call(self, x):
        x = self.embed(x)
        x = self.norm(x + self.ffn(x))  # residual
        return self.head(x)

model = SwiGLUModel(d_model=64, n_classes=10)
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

X_demo = np.random.randn(200, 32).astype(np.float32)
y_demo = np.random.randint(0, 10, 200)
history = model.fit(X_demo, y_demo, epochs=3, verbose=0)
print(f'SwiGLU model training OK — final loss: {history.history["loss"][-1]:.4f}')

## Section 8 — @tf.function Benchmark

In [ ]:
x_bench = tf.constant(np.random.randn(1000, 512).astype(np.float32))

activations_to_bench = {
    'ReLU':    tf.keras.activations.relu,
    'Sigmoid': tf.keras.activations.sigmoid,
    'Tanh':    tf.keras.activations.tanh,
    'GELU':    tf.keras.activations.gelu,
    'Swish':   tf.keras.activations.swish,
    'ELU':     tf.keras.activations.elu,
    'SELU':    tf.keras.activations.selu,
    'Softplus': tf.keras.activations.softplus,
    'Mish':    mish_tf,
}

# JIT compile each activation
compiled = {name: tf.function(fn) for name, fn in activations_to_bench.items()}

N_WARMUP, N_RUNS = 10, 100
timings = {}

for name, fn in compiled.items():
    for _ in range(N_WARMUP): fn(x_bench)  # warmup / tracing
    t0 = time.perf_counter()
    for _ in range(N_RUNS): fn(x_bench)
    timings[name] = (time.perf_counter() - t0) / N_RUNS * 1000

sorted_t = sorted(timings.items(), key=lambda kv: kv[1])
baseline = sorted_t[0][1]
print(f'Activation benchmarks on {x_bench.shape} tensor (tf.function JIT):')
print(f'{"Activation":<14} {"ms":<10} {"Relative"}')
print('-' * 36)
for name, t in sorted_t:
    bar = '█' * max(1, int(t / baseline))
    print(f'{name:<14} {t:<10.4f} {bar}')

## Section 9 — End-to-End: Fashion-MNIST Activation Comparison

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

activation_configs = {
    'relu':  'relu',
    'gelu':  'gelu',
    'swish': 'swish',
    'elu':   'elu',
    'selu':  'selu',
}

val_acc_histories = {}

for act_name, act_str in activation_configs.items():
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(512, activation=act_str, input_shape=(784,)),
        tf.keras.layers.Dense(512, activation=act_str),
        tf.keras.layers.Dense(256, activation=act_str),
        tf.keras.layers.Dense(10)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    hist = model.fit(
        X_train, y_train,
        epochs=10,
        validation_split=0.1,
        batch_size=256,
        verbose=0
    )
    val_acc = hist.history['val_accuracy']
    val_acc_histories[act_name] = val_acc
    print(f'{act_name:<8}: final val acc = {val_acc[-1]:.4f}')

plt.figure(figsize=(9, 5))
for name, hist in val_acc_histories.items():
    plt.plot(range(1, 11), hist, 'o-', label=name, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy')
plt.title('Fashion-MNIST MLP: Activation Function Comparison (TensorFlow)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()